<a href="https://colab.research.google.com/github/AN2DL-Challenge-26-27/AN2DL-Challenge-26-27/blob/Edoardo's-Branch/AN2DL_Challenge_26_27.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade kaggle

from google.colab import userdata
import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

KAGGLE_token = userdata.get("KAGGLE_TOKEN")

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
    f.write(KAGGLE_token)

# Imposta i permessi di sicurezza
os.chmod(os.path.expanduser("~/.kaggle/access_token"), 0o600)

print("Token configurato correttamente!")

!kaggle competitions download -c an2dl2627 -p /content/

# Estrai il file zip
import zipfile
import glob

zip_path = glob.glob('/content/*.zip')[0]
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('/content/')

print("Fatto!")

: 

# SPAGHETTI-1 — Dataset study (EDA)

Goal: understand the data before training any model. The organisers warn that **train and test may look different** and that **not every training entry can be trusted**, so besides the basic statistics we check integrity, domain shift, possible shortcuts, near-duplicates and overlays.

## 1. Setup and configuration
All constants used by the following cells are defined here, in a single place.

In [ ]:
# ---------------------------------------------------------------
# Global imports and configuration shared by all the cells below.
# ---------------------------------------------------------------
import json
from concurrent.futures import ThreadPoolExecutor
from hashlib import sha256
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

DATA_DIR = Path("/content")   # folder where the Kaggle zip was extracted (contains train/, test/, CSVs)
SEED = 42                     # fixed seed -> the random samples are reproducible
N_WORKERS = 8                 # threads used to read the images in parallel
IMAGE_SIZE = (256, 256)       # (width, height) expected for every PNG

# CSV listing the ids of each split (test ids come from the sample submission)
SPLIT_CSVS = {"train": "train.csv", "test": "sample_submission.csv"}
EXPECTED_COUNTS = {"train": 4328, "test": 14736}

CLASS_NAMES = {
    0: "Spaghetti alla rinfusa",   # irregular / disturbed
    1: "Gemelli",                  # merging pair
    2: "Gnocchi",                  # round smooth
    3: "Ravioli",                  # in-between smooth
    4: "Penne",                    # cigar-shaped smooth
    5: "Fusilli con la sbarra",    # barred spiral
    6: "Fusilli stretti",          # tight spiral
    7: "Riccioli",                 # loose spiral
    8: "Linguine",                 # edge-on disk
    9: "Lasagna",                  # edge-on disk with bulge
}

## 2. Manifest and file layout validation
Before analysing images we make sure the data is complete: correct CSV headers, well-formed and unique ids, labels in 0–9, the official counts (4,328 train / 14,736 test), and a one-to-one match between CSV ids and PNG files.

In [ ]:
# ---------------------------------------------------------------
# Read both CSVs into a single table (the "manifest") and check that
# CSVs and image folders are consistent.
# ---------------------------------------------------------------
def load_manifest(data_dir: Path) -> pd.DataFrame:
    """Return one table with columns id, label, split. Test labels are unknown, so they are set to NA."""
    splits = []
    for split, csv_name in SPLIT_CSVS.items():
        frame = pd.read_csv(data_dir / csv_name, dtype={"id": "string"})
        if list(frame.columns) != ["id", "label"]:
            raise ValueError(f"{csv_name}: expected the header id,label")
        # every id must look like train_00000 / test_00000
        if not frame["id"].str.fullmatch(rf"{split}_[0-9]{{5}}").fillna(False).all():
            raise ValueError(f"{csv_name}: missing or malformed image IDs")
        if frame.empty or frame["id"].duplicated().any():
            raise ValueError(f"{csv_name}: empty manifest or duplicate image IDs")
        if split == "train" and not frame["label"].isin(list(CLASS_NAMES)).all():
            raise ValueError(f"{csv_name}: expected integer class labels 0-9")
        if split == "test":
            frame["label"] = pd.NA
        frame["label"] = frame["label"].astype("Int64")     # integer type that also allows NA
        splits.append(frame.assign(split=split))
    return pd.concat(splits, ignore_index=True)


def validate_layout(data_dir: Path, manifest: pd.DataFrame) -> dict:
    """Check the expected number of rows and that every CSV id has exactly one PNG (and vice versa)."""
    summary = {}
    for split, expected_count in EXPECTED_COUNTS.items():
        folder = data_dir / split
        if not folder.is_dir():
            raise FileNotFoundError(f"Missing image directory: {folder}")
        ids = set(manifest.loc[manifest["split"] == split, "id"])
        file_ids = {path.stem for path in folder.glob("*.png") if path.is_file()}
        if len(ids) != expected_count:
            raise ValueError(f"{split}: expected {expected_count:,} CSV rows, found {len(ids):,}")
        missing, unexpected = sorted(ids - file_ids), sorted(file_ids - ids)
        if missing or unexpected:
            raise ValueError(f"{split}: {len(missing):,} missing PNGs {missing[:5]}, "
                             f"{len(unexpected):,} unexpected PNGs {unexpected[:5]}")
        summary[split] = {"rows": len(ids), "png_files": len(file_ids)}
    return summary


manifest = load_manifest(DATA_DIR)
print(json.dumps(validate_layout(DATA_DIR, manifest), indent=2))
manifest.head()

## 3. Image reader and per-image statistics
Every image is read once and described by a few numbers:

| Metric | Meaning |
|---|---|
| `mean`, `std` | brightness and contrast (0–255 scale) |
| `zero_fraction`, `saturated_fraction` | share of pure-black / pure-white values |
| `red_mean`, `green_mean`, `blue_mean` | average colour |
| `hf_energy` | mean difference between neighbouring pixels → amount of fine detail / noise |
| `center_mean`, `border_mean` | brightness of the central 64×64 window and of the borders |
| `colorfulness` | how coloured the image is (used to look for overlays) |
| `sha256` | fingerprint of the file bytes (exact duplicates) |

Two compact representations are also stored: a 64×64 thumbnail (for mean images) and a 32×32 high-pass descriptor (for near-duplicates).

In [ ]:
# ---------------------------------------------------------------
# Image reader + per-image statistics, computed in ONE pass over all
# 19,064 images (each file is read only once).
# ---------------------------------------------------------------
THUMB = 64      # side of the colour thumbnail kept in memory (mean images)
HP_SIDE = 32    # side of the high-pass descriptor used for the near-duplicate search

METRICS = ["mean", "std", "zero_fraction", "saturated_fraction", "red_mean", "green_mean",
           "blue_mean", "hf_energy", "center_mean", "border_mean", "colorfulness"]


def read_rgb_png(source) -> np.ndarray:
    """Decode a competition PNG, check it is a 256x256 RGB PNG and return a uint8 array (256, 256, 3)."""
    with Image.open(source) as image:
        if (image.format, image.mode, image.size) != ("PNG", "RGB", IMAGE_SIZE):
            raise ValueError(f"Expected 256x256 RGB PNG, found {image.format} {image.mode} {image.size}")
        return np.asarray(image)


_BOX = np.ones(5, np.float32) / 5   # 1-D averaging kernel (box blur of width 5)

def _blur_1d(v: np.ndarray) -> np.ndarray:
    return np.convolve(np.pad(v, 2, mode="edge"), _BOX, mode="valid")

def highpass_descriptor(gray: np.ndarray) -> np.ndarray:
    """Fine-detail fingerprint of an image: downscale to HP_SIDE x HP_SIDE, subtract a blurred
    copy (keeps stars/noise, removes the smooth galaxy shape), zero-mean and unit-norm."""
    f = gray.shape[0] // HP_SIDE
    small = gray.reshape(HP_SIDE, f, HP_SIDE, f).mean(axis=(1, 3))          # block average
    blurred = np.apply_along_axis(_blur_1d, 1, np.apply_along_axis(_blur_1d, 0, small))
    hp = (small - blurred).ravel()
    hp -= hp.mean()
    norm = np.linalg.norm(hp)
    return (hp / norm if norm > 0 else hp).astype(np.float32)


def inspect_image(path: Path):
    """Return (statistics dict, 64x64 thumbnail, high-pass descriptor) for one image.
    If the file is broken, the 'error' field is filled and the other values stay empty."""
    result = dict.fromkeys(["sha256", "error", *METRICS])
    thumb = np.zeros((THUMB, THUMB, 3), np.uint8)
    hp = np.zeros(HP_SIDE * HP_SIDE, np.float32)
    try:
        raw = path.read_bytes()
        result["sha256"] = sha256(raw).hexdigest()           # fingerprint of the file bytes
        pixels = read_rgb_png(BytesIO(raw)).astype(np.float32)
    except (OSError, ValueError, Image.DecompressionBombError) as exc:
        result["error"] = str(exc)
        return result, thumb, hp

    gray = pixels.mean(axis=2)
    r, g, b = pixels[..., 0], pixels[..., 1], pixels[..., 2]
    # hf_energy: mean absolute difference between neighbouring pixels (amount of fine detail/noise)
    hf = (np.abs(np.diff(gray, axis=0)).mean() + np.abs(np.diff(gray, axis=1)).mean()) / 2
    # colorfulness (Hasler & Suesstrunk): spread and mean of the opponent colour channels
    rg, yb = r - g, 0.5 * (r + g) - b
    colorfulness = np.hypot(rg.std(), yb.std()) + 0.3 * np.hypot(rg.mean(), yb.mean())
    edges = (gray[:16], gray[-16:], gray[:, :16], gray[:, -16:])   # 16-pixel border strips

    result.update(
        mean=float(pixels.mean()),                    # brightness
        std=float(pixels.std()),                      # contrast
        zero_fraction=float((pixels == 0).mean()),    # share of pure-black values
        saturated_fraction=float((pixels == 255).mean()),
        red_mean=float(r.mean()), green_mean=float(g.mean()), blue_mean=float(b.mean()),
        hf_energy=float(hf),
        center_mean=float(gray[96:160, 96:160].mean()),          # central 64x64 window
        border_mean=float(np.mean([e.mean() for e in edges])),
        colorfulness=float(colorfulness),
    )
    f = pixels.shape[0] // THUMB
    thumb = pixels.reshape(THUMB, f, THUMB, f, 3).mean(axis=(1, 3)).round().astype(np.uint8)
    return result, thumb, highpass_descriptor(gray)


# Parallel scan: threads overlap disk reading and PNG decoding
paths = [DATA_DIR / row.split / f"{row.id}.png" for row in manifest.itertuples()]
print(f"Scanning {len(paths):,} images with {N_WORKERS} workers...")
with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    outputs = list(pool.map(inspect_image, paths))

# Row i of stats / thumbs / hp_desc always refers to the same image (row i of manifest)
stats = pd.concat([manifest, pd.DataFrame([o[0] for o in outputs])], axis=1)
stats[METRICS] = stats[METRICS].astype(float)
thumbs = np.stack([o[1] for o in outputs])     # (N, 64, 64, 3) uint8, about 230 MB
hp_desc = np.stack([o[2] for o in outputs])    # (N, 1024) float32, about 80 MB
del outputs
stats.head()

## 4. Integrity checks
Unreadable files, (almost) uniform images and byte-identical duplicates. A duplicate shared between train and test, or with two different labels, would be a strong sign of an untrustworthy entry.

In [ ]:
# ---------------------------------------------------------------
# Integrity checks on the scanned images.
# ---------------------------------------------------------------
invalid = stats[stats["error"].notna()]          # files that could not be decoded or have a wrong format
valid = stats[stats["error"].isna()].copy()      # all the following analyses use only valid images
train = valid[valid["split"] == "train"]
test = valid[valid["split"] == "test"]

# Almost uniform images (pixel std < 2 on the 0-255 scale): probably empty or broken
low_contrast = valid[valid["std"] < 2.0]

# Byte-identical files (same SHA-256), also across splits and with conflicting labels
exact_dup = valid[valid["sha256"].duplicated(keep=False)].copy()
groups = exact_dup.groupby("sha256")
exact_dup["cross_split"] = groups["split"].transform("nunique") > 1
exact_dup["conflicting_labels"] = groups["label"].transform("nunique") > 1

print(f"Invalid images:            {len(invalid)}")
print(f"Low-contrast images:       {len(low_contrast)}")
print(f"Exact duplicate groups:    {exact_dup['sha256'].nunique()}")
print(f"  across train/test:       {exact_dup.loc[exact_dup['cross_split'], 'sha256'].nunique()}")
print(f"  with conflicting labels: {exact_dup.loc[exact_dup['conflicting_labels'], 'sha256'].nunique()}")
for name, table in (("Invalid", invalid), ("Low contrast", low_contrast), ("Exact duplicates", exact_dup)):
    if len(table):
        print(name); display(table.head(20))

## 5. Class distribution
The classes are imbalanced. The leaderboard metric is the **weighted F1**, where each class counts in proportion to its share of the test set, so frequent classes weigh more but rare ones are not irrelevant.

In [ ]:
# ---------------------------------------------------------------
# Number and percentage of training images per class.
# ---------------------------------------------------------------
counts = manifest.loc[manifest["split"] == "train", "label"].value_counts().reindex(list(CLASS_NAMES), fill_value=0)
dist = pd.DataFrame({"label": list(CLASS_NAMES),
                     "class_name": list(CLASS_NAMES.values()),
                     "count": counts.to_numpy()})
dist["percentage"] = 100 * dist["count"] / dist["count"].sum()
display(dist.round(2))

present = dist.loc[dist["count"] > 0, "count"]
print(f"Largest / smallest class ratio: {present.max() / present.min():.2f}")

fig, ax = plt.subplots(figsize=(11, 5), layout="constrained")
bars = ax.bar(dist["class_name"], dist["count"], color="steelblue")
ax.bar_label(bars, padding=3)
ax.tick_params(axis="x", labelrotation=35)
ax.set(ylabel="Images", title="Training class distribution")
ax.margins(y=0.15)
plt.show()

## 6. Domain shift: train vs test
New observations may not look like the training archive. If the two distributions differ, a model trained on train will see "unfamiliar" inputs at test time; the measured differences guide normalisation and data augmentation (e.g. random brightness/contrast reduction and blur).

In [ ]:
# ---------------------------------------------------------------
# Domain shift: compare the distribution of each statistic between
# train and test (table + overlapping normalised histograms).
# ---------------------------------------------------------------
domain = valid.groupby("split")[METRICS].agg(["mean", "std", "min", "max"])
display(domain.round(3))

# Relative difference of the means: (test - train) / train
rel_diff = 100 * (test[METRICS].mean() - train[METRICS].mean()) / train[METRICS].mean().abs()
display(rel_diff.round(1).rename("test vs train mean difference (%)").to_frame())

fig, axes = plt.subplots(2, 3, figsize=(14, 7), layout="constrained")
for ax, metric in zip(axes.flat, ("mean", "std", "hf_energy", "red_mean", "green_mean", "blue_mean")):
    bins = np.histogram_bin_edges(valid[metric], bins=50)   # same bins for both splits
    for split, frame in (("train", train), ("test", test)):
        ax.hist(frame[metric], bins=bins, density=True, alpha=0.55, label=split)
    ax.set(title=metric, ylabel="Density")
    ax.legend()
fig.suptitle("Train vs test image statistics")
plt.show()

## 7. Sample galleries
Visual inspection: random examples of each class, random test images, and the images with the highest high-frequency energy (candidates for noise or artefacts).

In [ ]:
# ---------------------------------------------------------------
# Image galleries. plot_grid draws one row of images per DataFrame in
# `rows`; each title shows class (or "Test"), id and a chosen metric.
# ---------------------------------------------------------------
def plot_grid(rows: list, width: int, title: str, metric: str = "hf_energy") -> None:
    fig, axes = plt.subplots(len(rows), width, figsize=(2.3 * width, 2.3 * len(rows)),
                             squeeze=False, layout="constrained")
    for axis_row, samples in zip(axes, rows):
        for ax in axis_row:
            ax.set_axis_off()                      # empty slots stay blank
        for ax, sample in zip(axis_row, samples.itertuples()):
            ax.imshow(read_rgb_png(DATA_DIR / sample.split / f"{sample.id}.png"))
            caption = CLASS_NAMES[int(sample.label)] if sample.split == "train" else "Test"
            ax.set_title(f"{caption}\n{sample.id} | {metric} {getattr(sample, metric):.1f}", fontsize=8)
    fig.suptitle(title)
    plt.show()

# 5 random examples per class (fixed seed -> always the same images)
class_examples = [train[train["label"] == label].sample(n=min(5, int((train["label"] == label).sum())),
                                                        random_state=SEED) for label in CLASS_NAMES]
plot_grid(class_examples, 5, "Training examples by class")

# 12 random test images
test_sample = test.sample(n=min(12, len(test)), random_state=SEED)
plot_grid([test_sample.iloc[:6], test_sample.iloc[6:]], 6, "Test examples")

# Images with the highest high-frequency energy (noisiest / most detailed)
plot_grid([train.nlargest(6, "hf_energy"), test.nlargest(6, "hf_energy")], 6,
          "Highest high-frequency energy (top: train, bottom: test)")

## 8. Per-class statistics (shortcut check)
If a class is systematically brighter or sharper than the others, a network may learn *brightness* instead of *shape*. Since the test set is darker, such a shortcut would fail at test time. The test distribution is shown as an extra box for comparison.

In [ ]:
# ---------------------------------------------------------------
# Distribution of brightness, contrast and high-frequency energy for every class; the whole test set is added as a last box for reference.
# Outliers are hidden (showfliers=False) to keep the boxes readable.
# ---------------------------------------------------------------
labels = list(CLASS_NAMES)
tick_names = [CLASS_NAMES[label] for label in labels] + ["TEST"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), layout="constrained")
for ax, metric in zip(axes, ("mean", "std", "hf_energy")):
    data = [train.loc[train["label"] == label, metric].to_numpy() for label in labels] + [test[metric].to_numpy()]
    ax.boxplot(data, showfliers=False)
    ax.set_xticks(range(1, len(tick_names) + 1), tick_names, rotation=60, ha="right")
    ax.set_title(metric)
fig.suptitle("Per-class image statistics (train) vs test")
plt.show()

# Median of each metric per class: a quick numeric view of the same plot
per_class_median = train.groupby("label")[["mean", "std", "hf_energy", "center_mean", "border_mean"]].median()
per_class_median.index = [CLASS_NAMES[label] for label in per_class_median.index]
display(per_class_median.round(2))

## 9. Mean image per class
Averaging all the images of a class shows its typical shape, size and centring at a glance (e.g. edge-on classes should appear as a thin stripe, round classes as a centred blob). It also tells us whether galaxies are always centred, which affects the choice of crop/translation augmentations.

In [ ]:
# ---------------------------------------------------------------
# Mean image of each class, computed on the 64x64 thumbnails.
# Each class image is rescaled to its own maximum so that the shape is visible; the last two panels (train vs test) share the same scale so that their brightness can be compared.
# ---------------------------------------------------------------
fig, axes = plt.subplots(2, 6, figsize=(15, 5.5), layout="constrained")
for ax, label in zip(axes.flat, CLASS_NAMES):
    mean_img = thumbs[train.index[train["label"] == label].to_numpy()].mean(axis=0)
    ax.imshow(mean_img / max(mean_img.max(), 1e-9))
    ax.set_title(CLASS_NAMES[label], fontsize=9)

split_means_img = {split: thumbs[frame.index.to_numpy()].mean(axis=0) for split, frame in (("train", train), ("test", test))}
common_max = max(img.max() for img in split_means_img.values())
for ax, (split, img) in zip(axes.flat[10:], split_means_img.items()):
    ax.imshow(img / common_max)
    ax.set_title(f"All {split} (same scale)", fontsize=9)
for ax in axes.flat:
    ax.set_axis_off()
fig.suptitle("Mean image per class")
plt.show()

## 10. Near-duplicates (high-pass correlation)
The SHA-256 check only finds byte-identical files; a re-saved, darkened or rotated copy has a different hash. A perceptual hash is not suitable here either: many galaxies are smooth round blobs on a black background and would get almost the same hash even when they are different objects.

We therefore compare the **fine details** of the images (background stars, noise), which are unique to each photograph: each image is reduced to 32×32, its blurred version is subtracted (*high-pass filter*), and the result is normalised. The correlation between two such descriptors is ≈1 for copies of the same photo and ≈0 for different photos, even if both show a smooth galaxy. All 8 rotations/flips are tested.

Why it matters: near-duplicates with **different labels** reveal label noise; duplicates between train and validation would inflate the validation score, so they must stay in the same fold.

In [ ]:
# ---------------------------------------------------------------
# Near-duplicate search: every TRAIN image is compared with every valid image (train + test) through the cosine similarity of the high-pass descriptors.
# Since descriptors are zero-mean and unit-norm, the dot product equals the correlation between the two images' fine details.
# ---------------------------------------------------------------
SIM_THRESHOLD = 0.90   # pairs above this similarity are reported; tune it with the histogram below
CHUNK = 256            # train images processed per block (limits memory usage)

def dihedral_variants(desc: np.ndarray) -> np.ndarray:
    """Return the 8 rotations/flips of each descriptor, shape (8, n, D).
    A galaxy rotated or mirrored is still the same image, so a copy can hide that way."""
    squares = desc.reshape(-1, HP_SIDE, HP_SIDE)
    variants = []
    for flipped in (squares, squares[:, :, ::-1]):
        for k in range(4):
            variants.append(np.rot90(flipped, k, axes=(1, 2)).reshape(len(squares), -1))
    return np.ascontiguousarray(np.stack(variants))

ref_pos = valid.index.to_numpy()                  # rows of hp_desc used as reference (train + test)
qry_pos = train.index.to_numpy()                  # rows used as queries (train only)
ref_t = np.ascontiguousarray(hp_desc[ref_pos].T)  # (D, R): transposed once for fast matrix products
variants = dihedral_variants(hp_desc[qry_pos])    # (8, Q, D)
self_col = np.searchsorted(ref_pos, qry_pos)      # column of each query inside the reference set

best_sim = np.empty(len(qry_pos), np.float32)     # similarity of each train image with its nearest neighbour
found_a, found_b, found_s = [], [], []
for start in range(0, len(qry_pos), CHUNK):
    stop = min(start + CHUNK, len(qry_pos))
    sim = variants[0, start:stop] @ ref_t
    for v in variants[1:]:                        # keep the best orientation for every pair
        np.maximum(sim, v[start:stop] @ ref_t, out=sim)
    sim[np.arange(stop - start), self_col[start:stop]] = -1.0   # ignore the image itself
    best_sim[start:stop] = sim.max(axis=1)
    rows, cols = np.nonzero(sim >= SIM_THRESHOLD)
    found_a.append(qry_pos[start + rows]); found_b.append(ref_pos[cols]); found_s.append(sim[rows, cols])

near = pd.DataFrame({"a": np.concatenate(found_a), "b": np.concatenate(found_b),
                     "similarity": np.concatenate(found_s)})
# a train-train pair is found twice (A->B and B->A): keep only one copy
near = near[~((stats.loc[near["b"], "split"].to_numpy() == "train") & (near["a"] > near["b"]))]
for side in ("a", "b"):
    near[f"id_{side}"] = stats.loc[near[side], "id"].to_numpy()
    # test labels are unknown (NA): they are shown as -1 so that labels can be compared safely
    near[f"label_{side}"] = stats.loc[near[side], "label"].fillna(-1).astype(int).to_numpy()
near["split_b"] = stats.loc[near["b"], "split"].to_numpy()
near["conflicting_labels"] = (near["split_b"] == "train") & (near["label_a"] != near["label_b"])
near = near.sort_values("similarity", ascending=False).reset_index(drop=True)

print(f"Pairs with similarity >= {SIM_THRESHOLD}: {len(near)} "
      f"(train-train {int((near['split_b'] == 'train').sum())}, "
      f"train-test {int((near['split_b'] == 'test').sum())}, "
      f"conflicting labels {int(near['conflicting_labels'].sum())})")

# Histogram of nearest-neighbour similarity: true duplicates form a separate group close to 1
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(best_sim, bins=100, log=True, color="steelblue")
ax.axvline(SIM_THRESHOLD, color="red", linestyle="--", label=f"threshold {SIM_THRESHOLD}")
ax.set(xlabel="Similarity with nearest neighbour", ylabel="Train images (log)",
       title="Nearest-neighbour similarity of train images")
ax.legend()
plt.show()

def show_pairs(pairs: pd.DataFrame, n: int = 8) -> None:
    """Show the first n pairs side by side (left = train image, right = its match)."""
    pairs = pairs.head(n)
    if pairs.empty:
        print("No pairs to show.")
        return
    fig, axes = plt.subplots(len(pairs), 2, figsize=(5, 2.5 * len(pairs)), squeeze=False, layout="constrained")
    for (ax_a, ax_b), pair in zip(axes, pairs.itertuples()):
        for ax, pos in ((ax_a, pair.a), (ax_b, pair.b)):
            row = stats.loc[pos]
            ax.imshow(read_rgb_png(DATA_DIR / row["split"] / f"{row['id']}.png"))
            caption = CLASS_NAMES[int(row["label"])] if row["split"] == "train" else "Test"
            ax.set_title(f"{caption} | {row['id']}", fontsize=8)
            ax.set_axis_off()
        ax_a.text(1.02, 0.5, f"sim {pair.similarity:.3f}", transform=ax_a.transAxes, fontsize=8)
    plt.show()

show_pairs(near)                                   # most similar pairs overall
show_pairs(near[near["conflicting_labels"]])       # same image, different labels -> label noise
display(near.head(20))

## 11. Overlays and anomalies
At least one training image contains an artificial overlay (an "alien" drawing). Galaxy photos are almost grey/yellowish, while a drawn overlay is usually strongly coloured, so we use the **colorfulness** metric (Hasler & Süsstrunk, 2003) to rank images. If overlays concentrate in some classes, a model could learn the overlay instead of the galaxy (*shortcut learning*). The flagged list is a starting point for **visual review**, not a final verdict.

In [ ]:
# ---------------------------------------------------------------
# Overlay / anomaly search based on colorfulness.
# A robust z-score is used: (x - median) / (1.4826 * MAD), where MAD is the median absolute deviation.
# Unlike mean/std it is not distorted by the outliers we are looking for.
# ---------------------------------------------------------------
COLOR_Z = 5.0   # images above this robust z-score are flagged (provisional, verify visually)

def robust_z(values: pd.Series) -> pd.Series:
    median = values.median()
    mad = (values - median).abs().median()
    return (values - median) / (1.4826 * mad + 1e-9)

# z-score computed over all valid images, so train and test are on the same scale
valid["colorfulness_z"] = robust_z(valid["colorfulness"])
train, test = valid[valid["split"] == "train"], valid[valid["split"] == "test"]

# Distribution of colorfulness per split
fig, ax = plt.subplots(figsize=(8, 4))
bins = np.histogram_bin_edges(valid["colorfulness"], bins=60)
for split, frame in (("train", train), ("test", test)):
    ax.hist(frame["colorfulness"], bins=bins, density=True, alpha=0.55, label=split)
ax.set(xlabel="colorfulness", ylabel="Density", yscale="log", title="Colorfulness by split (log scale)")
ax.legend()
plt.show()

# Most colourful images: overlays (e.g. the 'alien') should appear here
plot_grid([train.nlargest(6, "colorfulness"), train.nlargest(12, "colorfulness").iloc[6:]],
          6, "Most colourful TRAIN images", metric="colorfulness")
plot_grid([test.nlargest(6, "colorfulness"), test.nlargest(12, "colorfulness").iloc[6:]],
          6, "Most colourful TEST images", metric="colorfulness")

# Flagged images: how many per class (a class with many overlays = shortcut risk)
flag_train = train[train["colorfulness_z"] > COLOR_Z]
flag_test = test[test["colorfulness_z"] > COLOR_Z]
per_class = (flag_train["label"].value_counts().reindex(list(CLASS_NAMES), fill_value=0)
             .rename("flagged").to_frame())
per_class["class_size"] = dist.set_index("label")["count"]
per_class["flagged_%"] = 100 * per_class["flagged"] / per_class["class_size"]
per_class.index = [CLASS_NAMES[label] for label in per_class.index]
print(f"Flagged with z > {COLOR_Z}: {len(flag_train)} train ({100*len(flag_train)/len(train):.2f}%), "
      f"{len(flag_test)} test ({100*len(flag_test)/len(test):.2f}%)")
display(per_class.round(2))

## 12. Summary

In [ ]:
# ---------------------------------------------------------------
# Key numbers of the dataset study, collected in one dictionary (handy for the report).
# ---------------------------------------------------------------
def split_means(metric):
    """Mean of a per-image metric for each split, rounded."""
    return {split: round(float(frame[metric].mean()), 2) for split, frame in (("train", train), ("test", test))}

present = dist.loc[dist["count"] > 0, "count"]
summary = {
    "train_images": int(len(train)),
    "test_images": int(len(test)),
    "invalid_images": int(len(invalid)),
    "low_contrast_images": int(len(low_contrast)),
    "exact_duplicate_groups": int(exact_dup["sha256"].nunique()),
    "largest_to_smallest_class_ratio": round(float(present.max() / present.min()), 2),
    "mean_brightness": split_means("mean"),
    "mean_contrast_std": split_means("std"),
    "mean_hf_energy": split_means("hf_energy"),
    "mean_colorfulness": split_means("colorfulness"),
    "near_duplicate_pairs": {
        "threshold": SIM_THRESHOLD,
        "train_train": int((near["split_b"] == "train").sum()),
        "train_test": int((near["split_b"] == "test").sum()),
        "conflicting_labels": int(near["conflicting_labels"].sum()),
    },
    "colour_flagged": {"z_threshold": COLOR_Z, "train": int(len(flag_train)), "test": int(len(flag_test))},
}
print(json.dumps(summary, indent=2))

## 13. Cleaning decisions
Decisions based on the EDA results:

- **Near-duplicate thresholds.** In the train–train pairs the share of label conflicts drops from ~90% (the level expected for two *different* galaxies, 1 − Σ pᵢ² ≈ 88%) below 0.95, to ~20% in [0.97, 0.99) and ~2% above 0.99. Hence pairs ≥ 0.99 are true copies, [0.97, 0.99) is mixed, and below 0.95 they are only similar galaxies.
  - `group_id` (≥ 0.97): copies are grouped so that they never end up split between training and validation folds (grouping too much is harmless; missing a copy inflates validation).
  - `dup_id` (≥ 0.99): confident copies → keep one per group; drop groups whose copies have different labels.
  - Conflicts in [0.97, 0.99) are reviewed by eye (`MANUAL_DROP`).
- **Noisy images.** The ~8% of very colourful training images contain strong chromatic noise, present only in 4 classes and almost absent from the test set. They are kept, but flagged as `noisy`, because the noise could become a shortcut; it will be neutralised with augmentation.
- **Train–test copies** are used only to measure how test images differ from training ones (not to copy labels).

In [ ]:
# ---------------------------------------------------------------
# 13.1 Groups of copies via connected components ("union-find"):
# if A~B and B~C, then A, B and C are the same photo -> same group.
#   group_id (>= 0.97): used by StratifiedGroupKFold, so copies never end
#                       up split between training and validation
#   dup_id   (>= 0.99): confident copies, used by the cleaning rules (13.2)
# ---------------------------------------------------------------
GROUP_THRESHOLD = 0.97
DEDUP_THRESHOLD = 0.99
assert SIM_THRESHOLD <= GROUP_THRESHOLD, "re-run section 10 with SIM_THRESHOLD <= GROUP_THRESHOLD"

def connected_groups(positions: list, pairs: pd.DataFrame, threshold: float) -> pd.Series:
    """Return, for every position, the representative (smallest position) of its group."""
    parent = {p: p for p in positions}          # at the start every image is its own group

    def find(x):                                # follow the parents up to the group representative
        while parent[x] != x:
            parent[x] = parent[parent[x]]       # path compression: makes later searches faster
            x = parent[x]
        return x

    for a, b in pairs.loc[pairs["similarity"] >= threshold, ["a", "b"]].itertuples(index=False):
        root_a, root_b = find(int(a)), find(int(b))
        if root_a != root_b:                    # merge the two groups
            parent[max(root_a, root_b)] = min(root_a, root_b)
    return pd.Series([find(p) for p in positions], index=positions)

tt_pairs = near[near["split_b"] == "train"]     # only train-train pairs define training groups
positions = [int(p) for p in train.index]

clean = train[["id", "label"]].copy()           # one row per valid training image
clean["group_id"] = pd.factorize(connected_groups(positions, tt_pairs, GROUP_THRESHOLD))[0]
clean["dup_id"] = pd.factorize(connected_groups(positions, tt_pairs, DEDUP_THRESHOLD))[0]

for name in ("group_id", "dup_id"):
    sizes = clean.groupby(name).size()
    print(f"{name}: {int((sizes > 1).sum())} groups with copies, "
          f"{int(sizes[sizes > 1].sum())} images involved, largest group = {int(sizes.max())} images")
print("Group size distribution (group_id):")
display(clean.groupby("group_id").size().value_counts().sort_index().rename("groups").to_frame())

In [ ]:
# ---------------------------------------------------------------
# 13.2 Cleaning rules on the copies (dup_id groups, similarity >= 0.99):
#   - copies with the SAME label      -> keep only the first one (lowest id)
#   - copies with DIFFERENT labels    -> drop the whole group (true label unknown)
# Pairs in the uncertain band [0.97, 0.99) with different labels are shown
# for visual review: put the ids to remove in MANUAL_DROP and re-run.
# ---------------------------------------------------------------
MANUAL_DROP = []   # e.g. ["train_00713", "train_00854"], filled after looking at the pairs below

clean["drop_reason"] = ""
n_labels = clean.groupby("dup_id")["label"].transform("nunique")
group_size = clean.groupby("dup_id")["id"].transform("size")

conflict = n_labels > 1
clean.loc[conflict, "drop_reason"] = "conflicting copies (>= 0.99)"

extra_copy = (group_size > 1) & ~conflict & clean.duplicated("dup_id", keep="first")
clean.loc[extra_copy, "drop_reason"] = "extra copy (>= 0.99)"

manual = clean["id"].isin(MANUAL_DROP) & (clean["drop_reason"] == "")
clean.loc[manual, "drop_reason"] = "manual review"

# Uncertain band: same photo or just similar galaxies? Decide by eye.
review = tt_pairs[tt_pairs["conflicting_labels"]
                  & (tt_pairs["similarity"] >= GROUP_THRESHOLD)
                  & (tt_pairs["similarity"] < DEDUP_THRESHOLD)]
print(f"Conflicting pairs to review in [{GROUP_THRESHOLD}, {DEDUP_THRESHOLD}): {len(review)}")
show_pairs(review, n=min(len(review), 30))
display(review[["id_a", "label_a", "id_b", "label_b", "similarity"]])

In [ ]:
# ---------------------------------------------------------------
# 13.3 'noisy' flag: images with strong chromatic noise (colorfulness
# robust z-score > COLOR_Z, see section 11). They are KEPT, but the flag
# is stored to (a) evaluate validation F1 separately on noisy / clean
# images and (b) analyse errors. The noise itself will be neutralised
# with augmentation (random chromatic noise on all classes).
# ---------------------------------------------------------------
clean["noisy"] = valid.loc[clean.index, "colorfulness_z"].to_numpy() > COLOR_Z

noisy_table = pd.crosstab(clean["label"].map(CLASS_NAMES), clean["noisy"])
noisy_table = noisy_table.reindex(list(CLASS_NAMES.values()), fill_value=0)
noisy_table["noisy_%"] = 100 * noisy_table.get(True, 0) / noisy_table.sum(axis=1)
display(noisy_table.round(2))
print(f"Noisy training images: {int(clean['noisy'].sum())} of {len(clean)}")

In [ ]:
# ---------------------------------------------------------------
# 13.4 Train-test copies: the same galaxy seen in both domains.
# Ratios test/train of each statistic measure how the test images
# were transformed (darker? smoother? less colourful?).
# ---------------------------------------------------------------
cross = near[(near["split_b"] == "test") & (near["similarity"] >= GROUP_THRESHOLD)]
print(f"Train-test copies with similarity >= {GROUP_THRESHOLD}: {len(cross)}")

cmp_cols = ["mean", "std", "hf_energy", "colorfulness"]
train_side = stats.loc[cross["a"], cmp_cols].to_numpy()
test_side = stats.loc[cross["b"], cmp_cols].to_numpy()
ratios = pd.DataFrame(test_side / train_side, columns=[f"{c} test/train" for c in cmp_cols])
ratios.insert(0, "id_train", cross["id_a"].to_numpy())
ratios.insert(1, "id_test", cross["id_b"].to_numpy())
ratios.insert(2, "train_noisy", clean.set_index("id").loc[cross["id_a"], "noisy"].to_numpy())
ratios.insert(3, "similarity", cross["similarity"].to_numpy())
display(ratios.round(3))
print("Median ratio test/train:")
display(ratios[[f"{c} test/train" for c in cmp_cols]].median().round(3).to_frame("median"))

show_pairs(cross, n=len(cross))   # left = train version, right = test version

In [ ]:
# ---------------------------------------------------------------
# 13.5 Final cleaned training table: what was removed and why,
# class counts before/after, and CSV export (starting point for training).
# ---------------------------------------------------------------
print("Removed images by reason:")
display(clean.loc[clean["drop_reason"] != "", "drop_reason"].value_counts().rename("images").to_frame())

train_clean = clean[clean["drop_reason"] == ""][["id", "label", "group_id", "noisy"]].reset_index(drop=True)

before = clean["label"].value_counts().reindex(list(CLASS_NAMES), fill_value=0)
after = train_clean["label"].value_counts().reindex(list(CLASS_NAMES), fill_value=0)
class_change = pd.DataFrame({"class": list(CLASS_NAMES.values()),
                             "before": before.to_numpy(), "after": after.to_numpy()})
class_change["removed"] = class_change["before"] - class_change["after"]
display(class_change)
print(f"Training images: {len(clean)} -> {len(train_clean)} "
      f"({len(clean) - len(train_clean)} removed), {train_clean['group_id'].nunique()} groups")

# NOTE: /content is erased when the Colab runtime stops; copy the file to Drive if needed
CLEAN_CSV = DATA_DIR / "train_clean.csv"
train_clean.to_csv(CLEAN_CSV, index=False)
print(f"Saved {CLEAN_CSV}")